Установка зависимостей

In [ ]:
pip install -r requirements.txt

Импорт библиотек

In [ ]:
from __future__ import annotations
import os
import re
import io
import json
import csv
import itertools
import mimetypes
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Iterator
from datetime import datetime
from dataclasses import dataclass, field, asdict
import argparse

# PySpark imports
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    udf, col, input_file_name, size, array_join, explode, collect_list,
    sum as spark_sum, count, lit, when, expr, regexp_extract, struct, to_json,
    pandas_udf, PandasUDFType
)
from pyspark.sql.types import (
    StringType, IntegerType, BooleanType, ArrayType, MapType,
    StructType, StructField, BinaryType, LongType, DoubleType
)
from pyspark import SparkFiles, TaskContext
import pandas as pd
def safe_import(name: str):
    """Безопасный импорт библиотек на executor'ах"""
    try:
        return __import__(name)
    except Exception:
        return None
_libs = {}

def get_libs():
    """Ленивая загрузка библиотек на executor'е"""
    global _libs
    if not _libs:
        _libs = {
            'PyPDF2': safe_import('PyPDF2'),
            'pdfminer': safe_import('pdfminer'),
            'docx': safe_import('docx'),
            'bs4': safe_import('bs4'),
            'pandas': safe_import('pandas'),
            'PIL': safe_import('PIL'),
            'pytesseract': safe_import('pytesseract'),
            'chardet': safe_import('chardet'),
            'openpyxl': safe_import('openpyxl'),
        }
    return _libs

Spark

In [ ]:
def create_local_spark_session(app_name: str = "Local_PDN_Scanner", 
                               memory: str = "4g",
                               max_cores: int = None) -> SparkSession:
   
    builder = SparkSession.builder \
        .appName(app_name) \
        .master(f"local[{max_cores if max_cores else '*'}]") \
        .config("spark.driver.memory", memory) \
        .config("spark.driver.maxResultSize", "2g") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
        .config("spark.files.overwrite", "true") \
        .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem") \
        .config("spark.hadoop.fs.local.block.size", "33554432")  # 32MB
    
    # Дополнительные оптимизации для локальной работы
    builder = builder.config("spark.sql.shuffle.partitions", "8") \
        .config("spark.default.parallelism", "8")
    
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    
    print(f"Spark версия: {spark.version}")
    print(f"Режим: {spark.sparkContext.master}")
    print(f"Ядер доступно: {spark.sparkContext.defaultParallelism}")
    
    return spark

Категории персональных данных

In [ ]:
PDN_PATTERNS = {
    "PHONE": [
        r'(?:\+7|8)[\s\-]?\(?\d{3,4}\)?[\s\-]?\d{2,3}[\s\-]?\d{2}[\s\-]?\d{2}',
        r'(?:\+7|8)[\s\-]?\d{3}[\s\-]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}',
        r'\+\d{1,3}[\s\-]?\(?\d{1,4}\)?[\s\-]?\d{2,3}[\s\-]?\d{2}[\s\-]?\d{2}',
    ],
    "EMAIL": [
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
    ],
    "PASSPORT_RF": [
        r'\b\d{2}\s?\d{2}\s?\d{6}\b',
        r'\b\d{4}\s?\d{6}\b',
        r'(?:паспорт|passport|паспорт)\s*(?:рф|rf|РФ)?\s*[:№#]?\s*\d{2}\s?\d{2}\s?\d{6}',
    ],
    "SNILS": [
        r'\b\d{3}[\s\-]?\d{3}[\s\-]?\d{3}[\s\-]?\d{2}\b',
        r'(?:СНИЛС|снилс)\s*[:№#]?\s*\d{3}[\s\-]?\d{3}[\s\-]?\d{3}[\s\-]?\d{2}',
    ],
    "INN_PERSON": [
        r'\b\d{12}\b',
        r'(?:ИНН|инн)\s*(?:физ|фл|физическ)?\s*[:№#]?\s*\d{12}',
    ],
    "INN_LEGAL": [
        r'\b\d{10}\b',
        r'(?:ИНН|инн)\s*(?:юр|юридическ|организаци)?\s*[:№#]?\s*\d{10}',
    ],
    "BANK_CARD": [
        r'\b\d{4}[\s\-]?\d{4}[\s\-]?\d{4}[\s\-]?\d{4}\b',
        r'\b\d{4}[\s\-]?\d{6}[\s\-]?\d{5}\b',
    ],
    "BIK": [
        r'\b04\d{7}\b',
        r'(?:БИК|бик)\s*[:№#]?\s*04\d{7}',
    ],
    "BANK_ACCOUNT": [
        r'\b\d{20}\b',
        r'(?:р/с|расчетный\s*счет|счет)\s*[:№#]?\s*\d{20}',
    ],
    "BIRTH_DATE": [
        r'\b(?:0[1-9]|[12]\d|3[01])[\.\-/](?:0[1-9]|1[0-2])[\.\-/](?:19|20)\d{2}\b',
        r'(?:родился|родилась|рождени[яе]|birth|born|д\.р\.?)[\s:]*\d{2}[\.\-/]\d{2}[\.\-/]\d{4}',
    ],
    "ADDRESS": [
        r'(?:г\.|город|ул\.|улица|пр-т|проспект|д\.|дом|кв\.|квартира)\s*[\w\s\.,\-]+?\d+',
        r'\b\d{6}\b',
    ],
    "HEALTH": [
        r'\b(?:диагноз|заболевани[ея]|болезн[ьи]|симптом|лечени[ея]|врач|пациент|медицинск|больниц[аы]|поликлиник|инвалид|ВИЧ|СПИД|онкологи|диабет)\w*\b',
    ],
    "BIOMETRICS": [
        r'\b(?:биометри[яи]|отпечатк[ио]|пальц[ае]|радужн[аяой]\s*оболочк[аи]|сетчатк[аи]\s*глаз[а]?|голосов[аяой]\s*[сз]апис[ьи]?|распознавани[ея]\s*лиц[а]?|лицев[аяой]\s*биометри)\w*\b',
    ],
}

#УЗ-1
SPECIAL_CATEGORIES = {"HEALTH", "RELIGION", "POLITICS", "NATIONALITY", "BIOMETRICS"}

#УЗ-2/УЗ-3
GOV_IDS = {"PASSPORT_RF", "SNILS", "INN_PERSON", "INN_LEGAL", "DRIVER_LICENSE", "MRZ"}

#УЗ-2
PAYMENT_DATA = {"BANK_CARD", "BANK_ACCOUNT", "BIK", "CVV"}

#УЗ-3/УЗ-4
ORDINARY_PDN = {"FIO", "PHONE", "EMAIL", "BIRTH_DATE", "BIRTH_PLACE", "ADDRESS"}

Конфигурации и константы

In [ ]:
INCLUDE_EXTS = {'doc', 'docx', 'gif', 'html', 'ipynb', 'jpeg', 'jpg', 'pdf', 
                'php', 'png', 'rtf', 'xls', 'xlsx', 'txt', 'csv', 'json', 'tif', 'tiff'}

# Категории ПДн
PDN_CATEGORIES = ['обычные', 'государственные', 'платёжные', 'биометрические', 'специальные']

# Схема для результата анализа одного файла
ANALYSIS_RESULT_SCHEMA = StructType([
    StructField("path", StringType(), True),
    StructField("success", BooleanType(), True),
    StructField("ext", StringType(), True),
    StructField("text_length", IntegerType(), True),
    StructField("categories", MapType(StringType(), IntegerType()), True),
    StructField("total_hits", IntegerType(), True),
    StructField("uz", StringType(), True),
    StructField("error", StringType(), True)
])

Вспомогательные функции

In [ ]:
def detect_encoding(raw_bytes: bytes) -> str:
    libs = get_libs()
    if libs['chardet'] is None:
        return 'utf-8'
    try:
        res = libs['chardet'].detect(raw_bytes)
        enc = res.get('encoding') or 'utf-8'
        return enc
    except Exception:
        return 'utf-8'

def luhn_check(number: str) -> bool:
    digits = [int(d) for d in re.sub(r'\D', '', number)]
    if not (13 <= len(digits) <= 19):
        return False
    s = 0
    parity = len(digits) % 2
    for i, d in enumerate(digits):
        if i % 2 == parity:
            d *= 2
            if d > 9:
                d -= 9
        s += d
    return s % 10 == 0

def snils_valid(snils: str) -> bool:
    nums = re.sub(r'\D', '', snils)
    if len(nums) != 11:
        return False
    base = [int(x) for x in nums[:9]]
    check = int(nums[9:])
    s = sum((9 - i) * d for i, d in enumerate(base))
    if s < 100:
        c = s
    elif s in (100, 101):
        c = 0
    else:
        c = s % 101
        if c == 100:
            c = 0
    return c == check

def inn_valid(inn: str) -> bool:
    nums = re.sub(r'\D', '', inn)
    if len(nums) == 10:
        w = [2, 4, 10, 3, 5, 9, 4, 6, 8]
        c = sum(int(nums[i]) * w[i] for i in range(9)) % 11 % 10
        return c == int(nums[9])
    elif len(nums) == 12:
        w1 = [7, 2, 4, 10, 3, 5, 9, 4, 6, 8, 0]
        w2 = [3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8, 0]
        c1 = sum(int(nums[i]) * w1[i] for i in range(11)) % 11 % 10
        c2 = sum(int(nums[i]) * w2[i] for i in range(11)) % 11 % 10
        return c1 == int(nums[10]) and c2 == int(nums[11])
    return False

def has_context(text: str, idx: int, window: int, *keywords: str) -> bool:
    start = max(0, idx - window)
    end = min(len(text), idx + window)
    chunk = text[start:end].lower()
    return any(k.lower() in chunk for k in keywords)

Извлечение текста

In [ ]:
def extract_text_from_bytes(content: bytes, ext: str) -> str:
    libs = get_libs()
    if ext in {'txt', 'log', 'md', 'py', 'js', 'css', 'sql', 'php'}:
        try:
            enc = detect_encoding(content)
            return content.decode(enc, errors='ignore')
        except Exception:
            return ''
    elif ext in {'html', 'htm'}:
        try:
            txt = content.decode('utf-8', errors='ignore')
            if libs['bs4'] is not None:
                from bs4 import BeautifulSoup
                soup = BeautifulSoup(txt, 'html.parser')
                return soup.get_text(' ')
            return txt
        except Exception:
            return ''
    elif ext == 'pdf':
        text_parts = []
        if libs['pdfminer'] is not None:
            try:
                from pdfminer.high_level import extract_text as pdfminer_extract
                text = pdfminer_extract(io.BytesIO(content)) or ''
                if text.strip():
                    return text
            except Exception:
                pass
        if libs['PyPDF2'] is not None:
            try:
                reader = libs['PyPDF2'].PdfReader(io.BytesIO(content))
                for page in reader.pages:
                    try:
                        page_text = page.extract_text()
                        if page_text:
                            text_parts.append(page_text)
                    except Exception:
                        pass
                return '\n'.join(text_parts)
            except Exception:
                pass
        
        return '\n'.join(text_parts)
    
    elif ext == 'docx':
        if libs['docx'] is None:
            return ''
        try:
            from docx import Document
            doc = Document(io.BytesIO(content))
            parts = []
            for p in doc.paragraphs:
                if p.text:
                    parts.append(p.text)
            for tbl in doc.tables:
                for row in tbl.rows:
                    row_text = ' \t '.join(cell.text for cell in row.cells if cell.text)
                    if row_text:
                        parts.append(row_text)
            return '\n'.join(parts)
        except Exception:
            return ''
    elif ext == 'doc':
        try:
            text = content.decode('latin-1', errors='ignore')
            text = re.sub(r'[^\x20-\x7E\u0400-\u04FF]+', ' ', text)
            return re.sub(r'\s+', ' ', text)
        except Exception:
            return ''
    
    elif ext == 'rtf':
        try:
            raw = content.decode('utf-8', errors='ignore')
            raw = re.sub(r'\\[a-zA-Z]+-?\d*\s?', ' ', raw)
            raw = re.sub(r'[{}]', ' ', raw)
            return re.sub(r'\s+', ' ', raw)
        except Exception:
            return ''
    
    elif ext in {'xls', 'xlsx', 'xlsm'}:
        if libs['pandas'] is None:
            return ''
        try:
            df = libs['pandas'].read_excel(io.BytesIO(content), header=None, dtype=str)
            parts = []
            for _, row in df.iterrows():
                row_text = ' '.join(str(cell) for cell in row if pd.notna(cell))
                if row_text:
                    parts.append(row_text)
            return '\n'.join(parts)
        except Exception:
            return ''
    
    elif ext == 'csv':
        try:
            enc = detect_encoding(content)
            text = content.decode(enc, errors='ignore')
            return text.replace(',', ' | ').replace(';', ' | ')
        except Exception:
            return ''
    
    elif ext == 'json':
        try:
            data = json.loads(content.decode('utf-8', errors='ignore'))
            return json.dumps(data, ensure_ascii=False, indent=2)
        except Exception:
            return ''
    
    elif ext == 'ipynb':
        try:
            data = json.loads(content.decode('utf-8', errors='ignore'))
            parts = []
            for cell in data.get('cells', []):
                src = cell.get('source', [])
                if isinstance(src, list):
                    parts.append(''.join(src))
                elif isinstance(src, str):
                    parts.append(src)
            return '\n'.join(parts)
        except Exception:
            return ''
    
    elif ext in {'jpg', 'jpeg', 'png', 'gif', 'tif', 'tiff', 'bmp'}:
        if libs['PIL'] is None or libs['pytesseract'] is None:
            return ''
        try:
            from PIL import Image
            img = Image.open(io.BytesIO(content))
            if img.mode not in ('RGB', 'L'):
                img = img.convert('RGB')
            if img.width * img.height > 4000000:
                img.thumbnail((2000, 2000))
            return libs['pytesseract'].image_to_string(img, lang='rus+eng')
        except Exception:
            return ''
    
    return ''

Детекторы

In [ ]:
EMAIL_RE = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"(?:(?:\+7|8)\s*\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2})")
FIO_RE = re.compile(r"\b[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+)?\b")
DOB_RE = re.compile(r"\b(\d{2}[./]\d{2}[./]\d{4})\b")
INDEX_RE = re.compile(r"\b\d{6}\b")

SNILS_RE = re.compile(r"\b\d{3}-\d{3}-\d{3}\s?\d{2}\b|\b\d{11}\b")
INN10_RE = re.compile(r"(?<!\d)\d{10}(?!\d)")
INN12_RE = re.compile(r"(?<!\d)\d{12}(?!\d)")
PASSPORT_RE = re.compile(r"(?:(?<!\d)\d{2}\s?\d{2}\s?\d{6}(?!\d))")
MRZ_RE = re.compile(r"[P|V|C]<[A-Z<]{2}")

CARD_RE = re.compile(r"(?:(?:\d[ -]*?){13,19})")
CVV_RE = re.compile(r"\b(CVV|CVC|CVV2)\b", re.IGNORECASE)
RS_RE = re.compile(r"(?i)(?:р/с|расч[её]тн(?:ый)?\s+сч[её]т)[^\d]*(\d{20})")
BIK_RE = re.compile(r"(?i)бик[^\d]*(\d{9})")

BIOMETRIC_KEYS = [
    'биометр', 'отпечат', 'радуж', 'ирис', 'лицев', 'селфи', 'faceid', 
    'fingerprint', 'iris', 'voiceprint', 'голосов', 'геометрия лица'
]
SPECIAL_KEYS = [
    'диагноз', 'анамнез', 'инвалид', 'здоровь', 'медицин', 'психиатр', 
    'вич', 'религ', 'вероисповед', 'политическ', 'партия', 'интим', 'сексуаль'
]

def count_occurrences(pattern: re.Pattern, text: str) -> int:
    return len(list(pattern.finditer(text)))

def detect_categories(text: str) -> Dict[str, int]:
    t = text if isinstance(text, str) else ''
    low = t.lower()
    cats = {
        'обычные': 0,
        'государственные': 0,
        'платёжные': 0,
        'биометрические': 0,
        'специальные': 0
    }
    
    if not t:
        return cats
    
    cats['обычные'] += count_occurrences(EMAIL_RE, t)
    cats['обычные'] += count_occurrences(PHONE_RE, t)
    cats['обычные'] += min(5, count_occurrences(FIO_RE, t))
    
    for m in DOB_RE.finditer(t):
        if has_context(low, m.start(), 40, 'дата рождения', 'родил', 'д.р.', 'birth'):
            cats['обычные'] += 1
    
    for m in INDEX_RE.finditer(t):
        if has_context(low, m.start(), 40, 'ул', 'улица', 'просп', 'пер', 'дом', 'квартира', 'город', 'г.', 'адрес'):
            cats['обычные'] += 1
    
    for m in SNILS_RE.finditer(t):
        if snils_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in INN10_RE.finditer(t):
        if inn_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in INN12_RE.finditer(t):
        if inn_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in PASSPORT_RE.finditer(t):
        if has_context(low, m.start(), 50, 'паспорт', 'серия', 'номер', 'код подразделения'):
            cats['государственные'] += 1
    
    if MRZ_RE.search(t):
        cats['государственные'] += 1
    
    for m in CARD_RE.finditer(t):
        raw = m.group(0)
        digits = re.sub(r'\D', '', raw)
        if 13 <= len(digits) <= 19 and luhn_check(raw):
            if has_context(low, m.start(), 40, 'visa', 'mastercard', 'карта', 'cvv', 'cvc', 'номер карты'):
                cats['платёжные'] += 1
    
    for m in RS_RE.finditer(t):
        cats['платёжные'] += 1
    
    for m in BIK_RE.finditer(t):
        cats['платёжные'] += 1
    
    if CVV_RE.search(t):
        cats['платёжные'] += 1
    
    if any(k in low for k in BIOMETRIC_KEYS):
        cats['биометрические'] += 1
    
    if any(k in low for k in SPECIAL_KEYS):
        cats['специальные'] += 1
    
    return cats

def estimate_uz(cats: Dict[str, int]) -> str:
    total = sum(cats.values())
    distinct = sum(1 for v in cats.values() if v > 0)
    has_special = cats['специальные'] > 0
    has_bio = cats['биометрические'] > 0
    has_pay = cats['платёжные'] > 0
    has_gov = cats['государственные'] > 0
    has_common = cats['обычные'] > 0
    
    if has_special or has_bio:
        return 'УЗ-1' if (total >= 5 or distinct >= 2) else 'УЗ-2'
    if has_pay or has_gov:
        return 'УЗ-2' if (total >= 5 or distinct >= 2) else 'УЗ-3'
    if has_common:
        return 'УЗ-3' if (total >= 5 or distinct >= 2) else 'УЗ-4'
    return 'нет ПДн'

Тессеракт

Елка

Поиск в файлах

In [ ]:
def analyze_file_content(file_path: str, content: bytes) -> Dict[str, Any]:
    result = {
        'path': file_path,
        'success': False,
        'ext': '',
        'text_length': 0,
        'categories': {},
        'total_hits': 0,
        'uz': 'нет ПДн',
        'error': None
    }
    
    if content is None:
        result['error'] = 'Пустое содержимое'
        return result
    
    path = Path(file_path)
    ext = path.suffix.lower().lstrip('.')
    result['ext'] = ext
    
    if ext not in INCLUDE_EXTS:
        result['error'] = f'Неподдерживаемый формат: {ext}'
        return result
    
    try:
        text = extract_text_from_bytes(content, ext)
        result['text_length'] = len(text)
        
        if not text:
            result['error'] = 'Не удалось извлечь текст'
            result['success'] = True 
            return result
        
        cats = detect_categories(text)
        result['categories'] = {k: v for k, v in cats.items() if v > 0}
        result['total_hits'] = sum(cats.values())
        result['uz'] = estimate_uz(cats)
        result['success'] = True
        
    except Exception as e:
        result['error'] = str(e)[:200]
    
    return result

Класс распределенного сканирования

In [ ]:
class SparkPDNScanner:
    def __init__(self, spark: SparkSession):
        self.spark = spark
        
        self.analyze_udf = udf(analyze_file_content, ANALYSIS_RESULT_SCHEMA)
    
    @classmethod
    def create_local(cls, data_path: str = None, master: str = "local[*]", memory: str = "4g") -> "SparkPDNScanner":
        """
        Создает локальный экземпляр сканера
        
        Args:
            data_path: Путь к папке с данными (для автоматической настройки)
            master: Режим работы Spark
            memory: Память для драйвера
        """
        builder = SparkSession.builder \
            .appName("PDN_Scanner") \
            .master(master) \
            .config("spark.driver.memory", memory) \
            .config("spark.driver.maxResultSize", "2g") \
            .config("spark.sql.adaptive.enabled", "true") \
            .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
            .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
            .config("spark.files.overwrite", "true") \
            .config("spark.sql.shuffle.partitions", "8") \
            .config("spark.default.parallelism", "8")
        
        # ИСПРАВЛЕНИЕ: Настройка для работы с локальной файловой системой
        if data_path:
            data_path = str(Path(data_path).absolute())
            builder = builder \
                .config("spark.sql.warehouse.dir", f"file://{data_path}/spark-warehouse") \
                .config("spark.hadoop.fs.defaultFS", "file:///")
        
        spark = builder.getOrCreate()
        spark.sparkContext.setLogLevel("WARN")
        
        print(f"Spark сессия создана. Версия: {spark.version}")
        print(f"Режим: {spark.sparkContext.master}")
        
        return cls(spark)
    
    @classmethod
    def create_cluster(cls, app_name: str = "PDN_Scanner") -> "SparkPDNScanner":
        spark = SparkSession.builder \
            .appName(app_name) \
            .getOrCreate()
        return cls(spark)
    
    def scan_directory(self, 
                   directory_path: str, 
                   recursive: bool = True,
                   sample_size: Optional[int] = None) -> DataFrame:

        ext_pattern = "*.{" + ",".join(INCLUDE_EXTS) + "}"
        
        read_opts = {
            "pathGlobFilter": ext_pattern,
            "recursiveFileLookup": str(recursive).lower()
        }
        
        from pathlib import Path
        dir_path = Path(directory_path)
        
        if not dir_path.exists():
            raise FileNotFoundError(f"Директория не найдена: {directory_path}")
        
        absolute_path = f"file://{dir_path.absolute()}"
        print(f"Сканирование директории: {absolute_path}")
        
        try:
            df = self.spark.read \
                .format("binaryFile") \
                .options(**read_opts) \
                .load(absolute_path)
        except Exception as e:
            print(f"Ошибка чтения директории: {e}")
            empty_df = self.spark.createDataFrame([], ANALYSIS_RESULT_SCHEMA)
            return empty_df
        
        if sample_size:
            df = df.limit(sample_size)
        
        analyzed_df = df.select(
            self.analyze_udf(col("path"), col("content")).alias("result")
        ).select("result.*")
        
        analyzed_df.cache()
        file_count = analyzed_df.count()
        print(f"Найдено и проанализировано файлов: {file_count}")
        
        return analyzed_df
    
    def scan_file_list(self, file_paths: List[str]) -> DataFrame:
        def read_file(path: str) -> Optional[bytes]:
            try:
                with open(path, 'rb') as f:
                    return f.read()
            except Exception:
                return None
        
        read_udf = udf(read_file, BinaryType())
        
        paths_df = self.spark.createDataFrame([(p,) for p in file_paths], ["path"])
        files_df = paths_df.select(col("path"), read_udf(col("path")).alias("content"))
        
        analyzed_df = files_df.select(
            self.analyze_udf(col("path"), col("content")).alias("result")
        ).select("result.*")
        
        return analyzed_df
    
    def get_report_df(self, analyzed_df: DataFrame) -> DataFrame:
        report_df = analyzed_df.select(
            col("path").alias("путь"),
            expr("""
                concat_ws(', ', 
                    transform(
                        filter(map_entries(categories), x -> x.value > 0),
                        x -> concat(x.key, ':', cast(x.value as string))
                    )
                )
            """).alias("категории ПН"),
            col("total_hits").alias("количество находок"),
            col("uz").alias("УЗ"),
            col("ext").alias("формат файла")
        )
        
        return report_df
    
    def save_report_csv(self, analyzed_df: DataFrame, output_path: str) -> str:
        report_df = self.get_report_df(analyzed_df)
        
        report_df.coalesce(1) \
            .write \
            .mode("overwrite") \
            .option("header", "true") \
            .option("encoding", "UTF-8") \
            .csv(output_path)
        
        return output_path
    
    def get_statistics(self, analyzed_df: DataFrame) -> Dict[str, Any]:
        total = analyzed_df.count()
        success = analyzed_df.filter(col("success") == True).count()
        with_pdn = analyzed_df.filter(col("total_hits") > 0).count()
        
        uz_stats = analyzed_df \
            .groupBy("uz") \
            .agg(count("*").alias("count")) \
            .collect()
        
        cat_totals = {}
        for cat in PDN_CATEGORIES:
            total_cat = analyzed_df \
                .agg(spark_sum(col(f"categories.{cat}")).alias("sum")) \
                .collect()[0]["sum"]
            cat_totals[cat] = total_cat or 0
        
        return {
            'total_files': total,
            'successful': success,
            'failed': total - success,
            'files_with_pdn': with_pdn,
            'protection_levels': {row['uz']: row['count'] for row in uz_stats},
            'category_totals': cat_totals,
            'total_pdn_hits': sum(cat_totals.values())
        }
    
    def stop(self):
        self.spark.stop()

Сохранение и сканирование

In [ ]:
def scan_root_spark(root: Path, scanner: Optional[SparkPDNScanner] = None) -> pd.DataFrame:
    if scanner is None:
        scanner = SparkPDNScanner.create_local(data_path=str(root))
    
    if not root.exists():
        raise FileNotFoundError(f"Директория не найдена: {root}")
    
    print(f"Начало сканирования: {root.absolute()}")
    results_df = scanner.scan_directory(str(root))
    report_df = scanner.get_report_df(results_df)
    
    return report_df.toPandas()

def save_csv_spark(results_df: DataFrame, output_path: str) -> str:
    return results_df.write \
        .mode("overwrite") \
        .option("header", "true") \
        .option("encoding", "UTF-8") \
        .csv(output_path)

Локальная папка

In [ ]:
def scan_data_folder(data_subfolder: str = None, 
                     sample_size: int = None,
                     memory: str = "4g") -> pd.DataFrame:
    project_root = Path.cwd()
    data_dir = project_root / "data"
    
    if data_subfolder:
        data_dir = data_dir / data_subfolder
    
    if not data_dir.exists():
        raise FileNotFoundError(f"Папка не найдена: {data_dir}")
    
    print(f"Project root: {project_root}")
    print(f"Data directory: {data_dir}")
    
    scanner = SparkPDNScanner.create_local(data_path=str(data_dir), memory=memory)
    
    try:
        results = scanner.scan_directory(
            directory_path=str(data_dir),
            recursive=True,
            sample_size=sample_size
        )
        
        report_df = scanner.get_report_df(results)
        
        stats = scanner.get_statistics(results)
        print(f"Всего файлов: {stats['total_files']}")
        print(f"Успешно обработано: {stats['successful']}")
        print(f"Ошибок: {stats['failed']}")
        print(f"Файлов с ПДн: {stats['files_with_pdn']}")
        print(f"Всего находок: {stats['total_pdn_hits']}")
        print(f"\nПо уровням защищенности:")
        for level, count in stats['protection_levels'].items():
            print(f"  {level}: {count}")
        
        return report_df.toPandas()
        
    finally:
        scanner.stop()


def save_report_to_data_folder(df: pd.DataFrame, filename: str = None) -> Path:
    project_root = Path.cwd()
    reports_dir = project_root / "data" / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)
    
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"pdn_report_{timestamp}.csv"
    
    output_path = reports_dir / filename
    df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Отчет сохранен: {output_path}")
    
    return output_path

Main

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Распределенный сканер ПДн на Spark")
    parser.add_argument("--input", "-i", help="Директория для сканирования (по умолчанию ./data)")
    parser.add_argument("--output", "-o", help="Путь для сохранения CSV отчета (по умолчанию ./data/reports/)")
    parser.add_argument("--sample", "-s", type=int, help="Ограничение количества файлов (для теста)")
    parser.add_argument("--no-recursive", action="store_true", help="Отключить рекурсивный обход")
    parser.add_argument("--local", action="store_true", help="Локальный режим (без кластера)")
    
    args = parser.parse_args()
    
    if args.input is None:
        args.input = str(Path.cwd() / "data")
    
    if args.output is None:
        reports_dir = Path.cwd() / "data" / "reports"
        reports_dir.mkdir(parents=True, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        args.output = str(reports_dir / f"pdn_report_{timestamp}")
    
    print(f"Запуск сканирования: {args.input}")
    print(f"Расширения: {', '.join(sorted(INCLUDE_EXTS))}")
    
    if args.local:
        scanner = SparkPDNScanner.create_local(data_path=args.input)
    else:
        scanner = SparkPDNScanner.create_cluster()
    
    try:
        results = scanner.scan_directory(
            directory_path=args.input,
            recursive=not args.no_recursive,
            sample_size=args.sample
        )
        
        scanner.save_report_csv(results, args.output)
        
        stats = scanner.get_statistics(results)
        
        print(f"\nВсего файлов: {stats['total_files']}")
        print(f"Успешно обработано: {stats['successful']}")
        print(f"Ошибок: {stats['failed']}")
        print(f"Файлов с ПДн: {stats['files_with_pdn']}")
        print(f"Всего находок ПДн: {stats['total_pdn_hits']}")
        print(f"\nРаспределение по УЗ: {stats['protection_levels']}")
        print(f"\nНаходки по категориям:")
        for cat, count in stats['category_totals'].items():
            if count > 0:
                print(f"  - {cat}: {count}")
        
        print(f"\nОтчет сохранен: {args.output}")
        
    finally:
        scanner.stop()